In [35]:
import pandas as pd
import numpy as np
import torch
import json
import os
import torch.nn.functional as F
from torcheval.metrics.functional import binary_f1_score
from os.path import commonprefix

from tqdm import tqdm
from torch import nn
from transformers import DistilBertForSequenceClassification, AdamW, DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader, TensorDataset, SequentialSampler
from timeit import default_timer as timer
from os import walk
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, precision_score, roc_curve, auc, brier_score_loss

In [47]:
M4_DATA_FOLDER_PATH = '../data/raw/m4-unified'
dir_path, dir_names, file_names = next(walk(M4_DATA_FOLDER_PATH))

df = pd.DataFrame(columns=['text', 'is_llm', 'domain', 'dataset_name', 'prompt'])

for dir in dir_names:
    dataset_folder_path, _, dataset_names = next(walk(os.path.join(dir_path, dir)))
    for dataset_name in dataset_names:
        temp_df = pd.read_json(path_or_buf=f'{dataset_folder_path}/{dataset_name}', lines=True)
        temp_df['domain'] = dir
        temp_df['dataset_name'] = Path(dataset_name).stem
        temp_df['author'] = Path(dataset_name).stem.split('_')[1]
        temp_df['is_llm'] = 0 if 'human' in dataset_name else 1
        print(dataset_name, 0 if 'human' in dataset_name else 1)
        df = pd.concat([df, temp_df], ignore_index=True)

arxiv_bloomz.jsonl 1
arxiv_chatGPT.jsonl 1
arxiv_cohere.jsonl 1
arxiv_davinci.jsonl 1
arxiv_flant5.jsonl 1
arxiv_human.jsonl 0
reddit_bloomz.jsonl 1
reddit_chatGPT.jsonl 1
reddit_cohere.jsonl 1
reddit_davinci.jsonl 1
reddit_dolly.jsonl 1
reddit_flant5.jsonl 1
reddit_human.jsonl 0
wikihow_bloomz.jsonl 1
wikihow_chatGPT.jsonl 1
wikihow_cohere.jsonl 1
wikihow_davinci.jsonl 1
wikihow_dolly2.jsonl 1
wikihow_human.jsonl 0
wikipedia_bloomz.jsonl 1
wikipedia_chatGPT.jsonl 1
wikipedia_cohere.jsonl 1
wikipedia_davinci.jsonl 1
wikipedia_dolly.jsonl 1
wikipedia_human.jsonl 0


In [ ]:
df['prompt_start'] = df['prompt'].str[:20]
grouped_df = df.groupby(by="prompt_start")
group_indices = grouped_df.groups
group_indices_df = pd.DataFrame([
    {'group_key': key, 'indices': list(indices)}
    for key, indices in group_indices.items()
])
group_sizes = grouped_df.size().reset_index(name='count')


,prompt_start,count
0,Gen erate wikihow ar,1
1,Generate a 150-220-w,3000
2,Generate a WikiHow a,1483
3,Generate an abstract,4485
4,Generate wikihow art,2999
5,"Please, generate wik",6000
6,Rephrase the abstrac,3000
7,Write a WikiHow arti,4517
8,Write a Wikipedia ar,11033
9,Write a long abstrac,3000


In [54]:
group_summary = grouped_df['prompt'].agg([
    ('row_count', 'count'),
    ('longest_common_prefix', lambda x: commonprefix(x.tolist()))
]).reset_index()
group_summary.drop(columns=['prompt_start'], inplace=True)
group_summary

,row_count,longest_common_prefix
0,1,Gen erate wikihow article with minimum 200 words from title 'How to Feel Better when You're on a Break from Your Boyfriend'
1,3000,Generate a 150-220-word abstract for work with title:
2,1483,Generate a WikiHow article content given a title and a headline. Use approximately 300 words.\nTitle:\n{title}\nHeadline:\n{headline}\nArticle content:\n
3,4485,Generate an abstract for
4,2999,Generate wikihow article with minimum 200 words from title 'How to
5,6000,"Please, generate wikihow article with length above 1000 characters from title 'How to"
6,3000,Rephrase the abstract of an article with title '
7,4517,Write a WikiHow article content given a title and a headline. Use approximately 300 words.\nTitle:\n
8,11033,"Write a Wikipedia article with the title """
9,3000,"Write a long abstract of a scientific paper from arXiv.org. Use approximately 200-400 words.\nTitle: ""{title}"".\nAbstract:\n"


In [53]:
df['text_length'] = df['text'].apply(len)
grouped_by_author_df = df.groupby(by="domain")
grouped_by_author_df['text'].agg(
    avg_word_count=lambda x: x.str.split().str.len().mean()
).reset_index()

,domain,avg_word_count
0,arxiv,174.449056
1,reddit,233.839857
2,wikihow,529.023556
3,wikipedia,367.302267
